In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from pydantic import BaseModel
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from typing import Annotated

In [5]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [7]:
class ChatState(BaseModel):
    messages : Annotated[list,add_messages] 

In [8]:
def ChatBotNode(state : ChatState) -> ChatState:
    res = llm.invoke(state.messages)
    state.messages = [res]
    return state

In [9]:
memory = InMemorySaver()

In [10]:
graph = StateGraph(ChatState)
graph.add_node("chatbot", ChatBotNode)

graph.add_edge(START,"chatbot")
graph.add_edge("chatbot", END)

In [14]:
finalgraph = graph.compile(checkpointer=memory)

In [19]:
res = finalgraph.invoke(
    {"messages":[{"role":"user","content":"what is my name"}]},
    {"configurable":{"thread_id":"abc123"}}
)

In [20]:
res["messages"][-1].content

'You told me your name is **Satyam\u202fKumar**.'